[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/077.%20The%20Dynamic%20%26%20Real-Time%20VRP/P77-Tier-6_executed.ipynb)

---


# 77. The Dynamic and Real-Time Vehicle Routing Problem
## Tier 6: Autonomous and Self-Optimizing Ecosystems

### Purpose and scope

This notebook illustrates a distributed, multi-agent approach to assigning delivery requests to vehicles. A request agent invites bids, vehicle agents evaluate requests using several criteria, and the request agent selects and negotiates with a vehicle. Zone agents maintain simple demand forecasts and can record repositioning suggestions. The teaching focus is this sequence of local interactions, not a proof of globally optimal vehicle routes.

The example uses 15 vehicles, four zones, and a peak-hour batch of 23 requests, including eight new arrivals. It also includes a separate urgent pharmaceutical request. Coordinates are points on a Euclidean plane; distances are coordinate units, and the simplified travel-time estimate is expressed in minutes. Time windows and request arrival times are minutes after midnight.

### Modeling assumptions and limitations

- The Contract Net Protocol is represented as a simplified request broadcast, bid evaluation, and assignment negotiation. Messages are represented by data structures, but a network transport, communication delay, retries, and concurrent negotiations are not simulated.
- A vehicle's remaining capacity is reduced when a request is assigned and is not restored during this batch. This treats capacity as a limited assignment budget; it does not model pickup and delivery load changes over a complete route.
- Each request contributes its origin and destination to the end of a vehicle's route. The bid's detour measure therefore compares the existing route with this appended route; it does not search all insertion positions or guarantee time-window feasibility for every stop on a multi-request route.
- Zone forecasts use request origins and exponential smoothing. Repositioning is recorded as a suggestion; vehicles are not physically moved.
- Traffic is represented by a scalar congestion factor. The example does not model road networks, service times, vehicle failures, or a working infrastructure-agent implementation. Traffic conditions are passed directly to the bidding process.
- Processing time measures Python execution time for the local simulation. It is not a measurement of network negotiation latency and should not be interpreted as evidence of real-world millisecond performance.

### How the assignment method works

The Contract Net Protocol is a coordination pattern: an agent announces a task, potential contractors submit bids, and the task manager selects a contractor. Here, each request agent is the task manager and each feasible vehicle agent is a potential contractor. This distributed decision process determines one request assignment at a time; it is not a centralized global optimization model.

For vehicle \(v\) and request \(r\), the bid is a weighted heuristic:

$$
B_{vr} = 0.30D_{vr} + 0.25(2 - T_{vr}) + 0.25U_{vr} + 0.20F_{vr}.
$$

Here, \(D_{vr}\) is the additional route distance under the appended-route assumption, \(T_{vr}\) is a time-window compliance score, \(U_{vr}\) is the vehicle's capacity utilization after the assignment, and \(F_{vr}\) is a simplified future-impact score. Lower bids are preferred. The coefficients are illustrative weights, not empirically calibrated values; the terms have different scales, so the bid should be read as a ranking heuristic rather than a physical cost. A confidence score is also calculated, and the request agent prefers bids with confidence at least 0.7 when any such bid exists.

Travel time to the request origin is estimated as

$$
\tau(p,q,c) = \lVert p-q \rVert_2\left(1 + 0.5c\right),
$$

where \(p\) and \(q\) are 2D coordinates, \(c\) is the congestion level, and \(\lVert p-q \rVert_2\) is Euclidean distance. The estimated arrival is the request's arrival time plus this travel time. A vehicle is eligible only when its remaining capacity is sufficient and its estimated arrival is no later than the end of the request's time window. If it arrives before the window opens, waiting is allowed; the bid's compliance score is based on whether arrival can meet the window's end.

For \(R\) requests and \(V\) vehicles, the bidding and feasibility checks require \(O(RV)\) vehicle evaluations, apart from the cost of evaluating each vehicle's route. This local assignment sequence does not enumerate all joint assignments, but it also provides no global-optimality guarantee. Outcomes can depend on the order in which requests are processed.

### Reading the results

The simulation reports whether each request was assigned, how many bids were received, and the local processing time. The plots and diagnostic summaries describe this particular run. A high assignment rate would not, by itself, demonstrate globally optimal routing, and a low rate should be investigated alongside feasibility, time windows, and remaining capacity. No performance result is assumed in advance; the cells below calculate results when run.


In [ ]:
from dataclasses import dataclass
from enum import Enum
from collections import deque
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Fixed seeds make the generated scenario repeatable across runs.
np.random.seed(42)
random.seed(42)

## 1. Define the agents' shared data

A message, bid, and service request are represented as data records so that agents can exchange structured information. The `AgentType` enumeration names the agent roles used by the design. In this simulation, vehicle, zone, and request agents have implementations; traffic conditions are supplied as input rather than managed by an infrastructure-agent class.

In [ ]:
class AgentType(Enum):
    """Roles available in the multi-agent design."""

    VEHICLE = 'vehicle'
    ZONE = 'zone'
    REQUEST = 'request'
    INFRASTRUCTURE = 'infrastructure'


@dataclass
class Message:
    """Structured information that agents could exchange."""

    sender_id: str
    receiver_id: str
    message_type: str
    content: Dict
    timestamp: float
    priority: int = 1


@dataclass
class Bid:
    """A vehicle's heuristic offer to serve one request."""

    vehicle_id: str
    request_id: str
    bid_value: float
    detour_cost: float
    time_compliance: float
    capacity_utilization: float
    future_impact: float
    confidence: float


@dataclass
class ServiceRequest:
    """A delivery task with a location pair and time window."""

    id: str
    origin: Tuple[int, int]
    destination: Tuple[int, int]
    demand: int
    time_window_early: float
    time_window_late: float
    priority: int
    arrival_time: float
    assigned_vehicle: Optional[str] = None
    status: str = 'pending'


print('Agent communication data structures are ready.')

## 2. Implement vehicle agents

A vehicle agent evaluates a request using route distance, time-window compliance, capacity utilization, and an estimate of future workload. The bid is a score for comparing candidate vehicles; the request agent, not the vehicle, chooses the winning bid. Route distance and travel-time calculations both use Euclidean geometry, so the location representation is consistent throughout the example.

In [ ]:
class VehicleAgent:
    """Vehicle with local state and a heuristic request-bidding rule."""

    def __init__(
        self,
        vehicle_id: str,
        initial_position: Tuple[int, int],
        capacity: int,
        zone_id: str,
    ):
        self.id = vehicle_id
        self.position = initial_position
        self.capacity = capacity
        self.remaining_capacity = capacity
        self.zone_id = zone_id
        self.current_route: List[Tuple[int, int]] = []
        self.assigned_requests: List[ServiceRequest] = []
        self.traffic_awareness: Dict = {}
        self.local_performance_metrics: Dict = {
            'total_distance': 0.0,
            'completed_requests': 0,
            'average_response_time': 0.0,
        }

    def calculate_bid(
        self,
        request: ServiceRequest,
        traffic_conditions: Dict,
    ) -> Bid:
        """Score this vehicle's offer using the notebook's weighted heuristic."""
        detour_cost = self._calculate_detour_cost(request)
        time_to_origin = self._estimate_travel_time(
            self.position,
            request.origin,
            traffic_conditions,
        )
        current_time = traffic_conditions.get('current_time', request.arrival_time)
        estimated_arrival = current_time + time_to_origin
        time_compliance = (
            1.0
            if estimated_arrival <= request.time_window_late
            else 0.5
        )

        capacity_utilization = (
            self.capacity - self.remaining_capacity + request.demand
        ) / self.capacity
        future_impact = self._estimate_future_impact(request)

        # Lower values are preferred; these illustrative weights combine heuristic criteria.
        w1, w2, w3, w4 = 0.30, 0.25, 0.25, 0.20
        bid_value = (
            w1 * detour_cost
            + w2 * (2.0 - time_compliance)
            + w3 * capacity_utilization
            + w4 * future_impact
        )
        confidence = min(
            1.0,
            (time_compliance + (1.0 - capacity_utilization)) / 2.0,
        )

        return Bid(
            vehicle_id=self.id,
            request_id=request.id,
            bid_value=bid_value,
            detour_cost=detour_cost,
            time_compliance=time_compliance,
            capacity_utilization=capacity_utilization,
            future_impact=future_impact,
            confidence=confidence,
        )

    def _calculate_detour_cost(self, request: ServiceRequest) -> float:
        """Measure added route distance when the request is appended."""
        current_route_cost = self._calculate_route_cost()
        new_route = self.current_route + [request.origin, request.destination]
        new_route_cost = self._calculate_route_cost_for_route(new_route)
        return new_route_cost - current_route_cost

    def _calculate_route_cost(self) -> float:
        """Return the Euclidean length of the current route."""
        return self._calculate_route_cost_for_route(self.current_route)

    def _calculate_route_cost_for_route(
        self,
        route: List[Tuple[int, int]],
    ) -> float:
        """Sum the straight-line distances between consecutive route points."""
        total_cost = 0.0
        current_position = self.position

        for next_position in route:
            total_cost += np.sqrt(
                (current_position[0] - next_position[0]) ** 2
                + (current_position[1] - next_position[1]) ** 2
            )
            current_position = next_position

        return float(total_cost)

    def _estimate_travel_time(
        self,
        origin: Tuple[int, int],
        destination: Tuple[int, int],
        traffic_conditions: Dict,
    ) -> float:
        """Estimate travel time in minutes using distance and congestion."""
        distance = np.sqrt(
            (origin[0] - destination[0]) ** 2
            + (origin[1] - destination[1]) ** 2
        )
        congestion_level = traffic_conditions.get('congestion', 0.5)
        traffic_factor = 1.0 + congestion_level * 0.5
        return float(distance * traffic_factor)

    def _estimate_future_impact(self, request: ServiceRequest) -> float:
        """Estimate how this assignment affects remaining service capacity."""
        workload_factor = len(self.assigned_requests) / 10.0
        capacity_factor = (
            self.capacity - self.remaining_capacity + request.demand
        ) / self.capacity
        return (workload_factor + capacity_factor) / 2.0

    def assign_request(self, request: ServiceRequest) -> None:
        """Commit a request after the vehicle accepts the assignment."""
        self.assigned_requests.append(request)
        self.remaining_capacity -= request.demand
        request.assigned_vehicle = self.id
        request.status = 'assigned'
        self.current_route.extend([request.origin, request.destination])


print('Vehicle agent implementation is ready.')

## 3. Implement zone agents

Zone agents organize vehicles geographically and update a smoothed estimate of local demand. Exponential smoothing combines the newest count with the previous estimate, so a single observation does not completely replace the forecast. The zone agent can record suggestions for idle vehicles when demand is rising; this example does not execute the repositioning.

In [ ]:
class ZoneAgent:
    """Manage a geographic region, its vehicles, and a simple demand forecast."""

    def __init__(self, zone_id: str, boundaries: Tuple[int, int, int, int]):
        self.id = zone_id
        self.boundaries = boundaries
        self.vehicles: List[VehicleAgent] = []
        self.demand_forecast: Dict = {}
        self.request_patterns: List = []
        self.coordination_history: List = []
        self.repositioning_suggestions: List = []

    def add_vehicle(self, vehicle: VehicleAgent) -> None:
        """Register a vehicle with this zone."""
        self.vehicles.append(vehicle)
        vehicle.zone_id = self.id

    def update_demand_forecast(self, recent_requests: List[ServiceRequest]) -> None:
        """Update the zone's request-count forecast using exponential smoothing."""
        zone_requests = [
            request
            for request in recent_requests
            if self._is_in_zone(request.origin)
        ]
        current_demand = len(zone_requests)
        historical_average = self.demand_forecast.get('average', current_demand)
        alpha = 0.3
        new_average = (
            alpha * current_demand
            + (1.0 - alpha) * historical_average
        )
        self.demand_forecast = {
            'average': new_average,
            'current': current_demand,
            'trend': current_demand - historical_average,
        }

    def guide_idle_vehicles(self) -> None:
        """Record repositioning suggestions for idle vehicles in rising-demand zones."""
        idle_vehicles = [
            vehicle
            for vehicle in self.vehicles
            if len(vehicle.assigned_requests) < 2
        ]
        if not idle_vehicles or self.demand_forecast.get('trend', 0) <= 0:
            return

        demand_center = self._calculate_demand_center()
        for vehicle in idle_vehicles:
            distance_to_center = np.sqrt(
                (vehicle.position[0] - demand_center[0]) ** 2
                + (vehicle.position[1] - demand_center[1]) ** 2
            )
            if distance_to_center > 20:
                self._suggest_repositioning(vehicle, demand_center)

    def coordinate_cross_zone_requests(
        self,
        request: ServiceRequest,
        other_zones: List['ZoneAgent'],
    ) -> Optional[VehicleAgent]:
        """Return the eligible vehicle with the lowest bid across these zones."""
        all_vehicles = []
        for zone in [self] + other_zones:
            all_vehicles.extend(zone.vehicles)

        eligible_vehicles = [
            vehicle
            for vehicle in all_vehicles
            if vehicle.remaining_capacity >= request.demand
        ]
        if not eligible_vehicles:
            return None

        traffic_conditions = {
            'congestion': 0.5,
            'current_time': request.arrival_time,
        }
        return min(
            eligible_vehicles,
            key=lambda vehicle: vehicle.calculate_bid(
                request,
                traffic_conditions,
            ).bid_value,
        )

    def _is_in_zone(self, position: Tuple[int, int]) -> bool:
        """Check whether a coordinate lies within this zone's boundaries."""
        x_min, y_min, x_max, y_max = self.boundaries
        return (
            x_min <= position[0] <= x_max
            and y_min <= position[1] <= y_max
        )

    def _calculate_demand_center(self) -> Tuple[int, int]:
        """Use the geometric center of the zone as a simple target location."""
        x_min, y_min, x_max, y_max = self.boundaries
        return ((x_min + x_max) // 2, (y_min + y_max) // 2)

    def _suggest_repositioning(
        self,
        vehicle: VehicleAgent,
        target: Tuple[int, int],
    ) -> None:
        """Record a target so the suggestion is observable without moving the vehicle."""
        suggestion = {'vehicle_id': vehicle.id, 'target': target}
        self.repositioning_suggestions.append(suggestion)
        self.coordination_history.append(suggestion)


print('Zone agent implementation is ready.')

## 4. Implement request agents and bidding

The request agent first filters vehicles by capacity and arrival-time feasibility. It then collects bids from the remaining vehicles, sorts them by bid value, and selects the lowest bid meeting the confidence threshold when possible. This is a sequential Contract Net-style assignment: one request is negotiated at a time, and the selected vehicle's capacity is checked again before commitment.

In [ ]:
class RequestAgent:
    """Represent one request and coordinate its bid-and-assignment process."""

    def __init__(self, request: ServiceRequest):
        self.request = request
        self.id = request.id
        self.bid_history: List[Bid] = []
        self.negotiation_rounds = 0
        self.max_negotiation_rounds = 3
        self.assignment_status = 'searching'

    def initiate_bidding_process(
        self,
        vehicle_agents: List[VehicleAgent],
        traffic_conditions: Dict,
    ) -> List[Bid]:
        """Request bids only from vehicles that pass the feasibility checks."""
        bids = []
        feasible_vehicles = self._find_feasible_vehicles(
            vehicle_agents,
            traffic_conditions,
        )
        for vehicle in feasible_vehicles:
            bid = vehicle.calculate_bid(self.request, traffic_conditions)
            bids.append(bid)
            self.bid_history.append(bid)
        return bids

    def evaluate_bids(self, bids: List[Bid]) -> Optional[str]:
        """Choose a vehicle ID, preferring a sufficiently confident low bid."""
        if not bids:
            return None

        sorted_bids = sorted(bids, key=lambda bid: bid.bid_value)
        for bid in sorted_bids:
            if bid.confidence >= 0.7:
                return bid.vehicle_id
        return sorted_bids[0].vehicle_id

    def negotiate_assignment(
        self,
        selected_vehicle_id: Optional[str],
        vehicle_agents: List[VehicleAgent],
    ) -> bool:
        """Confirm the selected vehicle still exists and has sufficient capacity."""
        if selected_vehicle_id is None:
            return False

        selected_vehicle = next(
            (
                vehicle
                for vehicle in vehicle_agents
                if vehicle.id == selected_vehicle_id
            ),
            None,
        )
        if selected_vehicle is None:
            return False
        if selected_vehicle.remaining_capacity < self.request.demand:
            return False

        selected_vehicle.assign_request(self.request)
        self.assignment_status = 'assigned'
        return True

    def _find_feasible_vehicles(
        self,
        vehicle_agents: List[VehicleAgent],
        traffic_conditions: Dict,
    ) -> List[VehicleAgent]:
        """Apply capacity and pickup time-window checks before requesting bids."""
        feasible_vehicles = []
        current_time = traffic_conditions.get(
            'current_time',
            self.request.arrival_time,
        )

        for vehicle in vehicle_agents:
            if vehicle.remaining_capacity < self.request.demand:
                continue

            travel_time = vehicle._estimate_travel_time(
                vehicle.position,
                self.request.origin,
                traffic_conditions,
            )
            estimated_arrival = current_time + travel_time
            if estimated_arrival <= self.request.time_window_late:
                feasible_vehicles.append(vehicle)

        return feasible_vehicles


print('Request agent implementation is ready.')

## 5. Coordinate the distributed system

The system object stores the agents and coordinates each request's bidding and assignment sequence. Wall-clock timing is kept separate from simulated time: simulated times are minutes after midnight, while `time.perf_counter()` measures local Python execution duration. The reported processing time therefore describes computation in this notebook, not travel time or communication delay.

In [ ]:
class DistributedDVRPSystem:
    """Coordinate local agents in a simplified distributed routing simulation."""

    def __init__(self, num_zones: int = 4):
        self.zone_agents: List[ZoneAgent] = []
        self.vehicle_agents: List[VehicleAgent] = []
        self.request_agents: List[RequestAgent] = []
        self.message_queue: deque = deque()
        self.system_metrics: Dict = {
            'total_requests_processed': 0,
            'average_response_time': 0.0,
            'vehicle_utilization': 0.0,
            'negotiation_overhead': 0.0,
        }
        self._initialize_zones(num_zones)

    def _initialize_zones(self, num_zones: int) -> None:
        """Create the zone grid used by the metropolitan example."""
        if num_zones < 1 or num_zones > 4:
            raise ValueError('The 2-by-2 example supports between one and four zones.')

        zone_size = 50
        for index in range(num_zones):
            row = index // 2
            column = index % 2
            x_min = column * zone_size
            y_min = row * zone_size
            boundaries = (
                x_min,
                y_min,
                x_min + zone_size,
                y_min + zone_size,
            )
            self.zone_agents.append(ZoneAgent(f'zone_{index}', boundaries))

    def add_vehicles(self, vehicles_config: List[Dict]) -> None:
        """Create vehicles and register each with the zone containing its position."""
        for config in vehicles_config:
            vehicle = VehicleAgent(
                vehicle_id=config['id'],
                initial_position=config['position'],
                capacity=config['capacity'],
                zone_id=config.get('zone_id', 'zone_0'),
            )
            self.vehicle_agents.append(vehicle)

            for zone in self.zone_agents:
                if zone._is_in_zone(vehicle.position):
                    zone.add_vehicle(vehicle)
                    break
            else:
                raise ValueError(
                    f'Vehicle {vehicle.id} is outside every configured zone.'
                )

    def process_request(self, request: ServiceRequest) -> Dict:
        """Run bidding, selection, and assignment negotiation for one request."""
        start_time = time.perf_counter()
        request_agent = RequestAgent(request)
        self.request_agents.append(request_agent)

        traffic_conditions = {
            'congestion': random.uniform(0.1, 0.9),
            'current_time': request.arrival_time,
        }
        bids = request_agent.initiate_bidding_process(
            self.vehicle_agents,
            traffic_conditions,
        )
        selected_vehicle_id = request_agent.evaluate_bids(bids)
        assignment_success = request_agent.negotiate_assignment(
            selected_vehicle_id,
            self.vehicle_agents,
        )
        processing_time = time.perf_counter() - start_time

        self.system_metrics['total_requests_processed'] += 1
        self.system_metrics['negotiation_overhead'] += processing_time
        self._update_system_metrics()

        return {
            'request_id': request.id,
            'assigned_vehicle': selected_vehicle_id,
            'assignment_success': assignment_success,
            'processing_time': processing_time,
            'bids_received': len(bids),
            'negotiation_rounds': request_agent.negotiation_rounds,
        }

    def optimize_system_performance(self) -> None:
        """Update zone forecasts and record local repositioning suggestions."""
        recent_requests = [
            agent.request for agent in self.request_agents[-10:]
        ]
        for zone in self.zone_agents:
            zone.update_demand_forecast(recent_requests)
            zone.guide_idle_vehicles()
        self._update_system_metrics()

    def _update_system_metrics(self) -> None:
        """Recalculate average local processing time and assigned capacity share."""
        total_requests = self.system_metrics['total_requests_processed']
        if total_requests > 0:
            self.system_metrics['average_response_time'] = (
                self.system_metrics['negotiation_overhead']
                / total_requests
                * 1000.0
            )

        total_capacity = sum(
            vehicle.capacity for vehicle in self.vehicle_agents
        )
        used_capacity = sum(
            vehicle.capacity - vehicle.remaining_capacity
            for vehicle in self.vehicle_agents
        )
        if total_capacity > 0:
            self.system_metrics['vehicle_utilization'] = (
                used_capacity / total_capacity
            )


print('Distributed DVRP system implementation is ready.')

## 6. Create the metropolitan scenario

The zone grid spans coordinates from 0 to 100 in each dimension. Vehicles are sampled within their assigned 50-by-50 regions, with the original scenario's 15-vehicle distribution. The fixed random seed makes these generated positions and capacities repeatable.

In [ ]:
def create_metropolitan_scenario() -> DistributedDVRPSystem:
    """Create the 15-vehicle metropolitan scenario across four zones."""
    print('Creating the metropolitan delivery scenario')
    print('=' * 48)

    system = DistributedDVRPSystem(num_zones=4)
    vehicles_config = []
    vehicle_id = 0

    for zone_index in range(4):
        number_of_vehicles = 4 if zone_index < 3 else 3
        for _ in range(number_of_vehicles):
            x_min = (zone_index % 2) * 50
            y_min = (zone_index // 2) * 50
            position = (
                random.randint(x_min, x_min + 49),
                random.randint(y_min, y_min + 49),
            )
            vehicles_config.append(
                {
                    'id': f'vehicle_{vehicle_id}',
                    'position': position,
                    'capacity': random.randint(15, 25),
                    'zone_id': f'zone_{zone_index}',
                }
            )
            vehicle_id += 1

    system.add_vehicles(vehicles_config)
    print(
        f'Created {len(system.vehicle_agents)} vehicles across '
        f'{len(system.zone_agents)} zones.'
    )
    for zone in system.zone_agents:
        print(
            f'{zone.id}: {len(zone.vehicles)} vehicles; '
            f'boundaries {zone.boundaries}'
        )
    return system


system = create_metropolitan_scenario()

## 7. Simulate peak-hour demand

The peak-hour example creates 15 existing requests and eight new arrivals. Each request is processed in sequence, so assignments made for earlier requests affect the remaining capacity available to later ones. Simulated request times are in minutes after midnight; processing durations are measured separately in seconds and converted to milliseconds for display.

In [ ]:
def simulate_peak_hour_scenario(system: DistributedDVRPSystem) -> Dict:
    """Generate and process 23 requests, including eight new arrivals."""
    print('\nSimulating the peak-hour scenario (12:30 PM)')
    print('=' * 48)

    requests = []
    current_time = 12.5 * 60

    for index in range(15):
        request = ServiceRequest(
            id=f'req_existing_{index}',
            origin=(random.randint(0, 99), random.randint(0, 99)),
            destination=(random.randint(0, 99), random.randint(0, 99)),
            demand=random.randint(1, 4),
            time_window_early=current_time + random.uniform(-30, 30),
            time_window_late=current_time + random.uniform(30, 120),
            priority=random.choices([1, 2, 3], weights=[0.2, 0.5, 0.3])[0],
            arrival_time=current_time - random.uniform(5, 60),
        )
        requests.append(request)

    for index in range(8):
        request = ServiceRequest(
            id=f'req_new_{index}',
            origin=(random.randint(0, 99), random.randint(0, 99)),
            destination=(random.randint(0, 99), random.randint(0, 99)),
            demand=random.randint(1, 4),
            time_window_early=current_time + random.uniform(15, 45),
            time_window_late=current_time + random.uniform(45, 120),
            priority=random.choices([1, 2, 3], weights=[0.3, 0.4, 0.3])[0],
            arrival_time=current_time,
        )
        requests.append(request)

    new_arrivals = sum(
        request.arrival_time == current_time for request in requests
    )
    print(f'Generated {len(requests)} requests, including {new_arrivals} new arrivals.')

    processing_results = []
    for index, request in enumerate(requests, start=1):
        print(f'\nProcessing request {index}/{len(requests)}: {request.id}')
        result = system.process_request(request)
        processing_results.append(result)
        status = 'ASSIGNED' if result['assignment_success'] else 'NOT ASSIGNED'
        print(
            f"  {status}; selected vehicle: {result['assigned_vehicle']}; "
            f"local processing time: {result['processing_time'] * 1000:.3f} ms"
        )
        print(f"  Bids received: {result['bids_received']}")

    print('\nUpdating zone forecasts and repositioning suggestions.')
    system.optimize_system_performance()

    return {
        'requests_processed': len(requests),
        'successful_assignments': sum(
            result['assignment_success'] for result in processing_results
        ),
        'processing_results': processing_results,
        'system_metrics': system.system_metrics.copy(),
    }


simulation_results = simulate_peak_hour_scenario(system)

## 8. Simulate an urgent pharmaceutical request

This example makes the assignment sequence visible: the request agent solicits bids, evaluates them, and asks the selected vehicle to confirm the assignment. The time window is represented consistently in minutes after midnight. The priority field describes the request, but it is not part of the bid formula; the current bidding rule therefore does not give urgent requests an additional numerical preference.

In [ ]:
def simulate_pharmaceutical_urgency_scenario(
    system: DistributedDVRPSystem,
) -> Dict:
    """Run the bid-and-assignment sequence for an urgent delivery request."""
    print('\nSimulating the urgent pharmaceutical delivery scenario')
    print('=' * 54)

    current_time = 12.5 * 60
    urgent_request = ServiceRequest(
        id='pharmacy_urgent_001',
        origin=(25, 35),
        destination=(65, 75),
        demand=2,
        time_window_early=12.5 * 60,
        time_window_late=13.0 * 60,
        priority=1,
        arrival_time=current_time,
    )

    print(f'Request: {urgent_request.origin} to {urgent_request.destination}')
    print(
        'Time window: '
        f'[{urgent_request.time_window_early / 60:.1f}, '
        f'{urgent_request.time_window_late / 60:.1f}] hours after midnight'
    )
    print('Priority: high (pharmaceutical delivery)')

    negotiation_log = []
    request_agent = RequestAgent(urgent_request)
    traffic_conditions = {
        'congestion': 0.6,
        'current_time': current_time,
    }

    print('\nStep 1: The request agent solicits bids.')
    bids = request_agent.initiate_bidding_process(
        system.vehicle_agents,
        traffic_conditions,
    )
    print(f'Bids received: {len(bids)}')
    negotiation_log.append(f'Bidding initiated: {len(bids)} bids received')

    print('\nStep 2: The request agent evaluates bids.')
    sorted_bids = sorted(bids, key=lambda bid: bid.bid_value)[:3]
    for rank, bid in enumerate(sorted_bids, start=1):
        print(
            f'  {rank}. {bid.vehicle_id}: bid={bid.bid_value:.3f}, '
            f'confidence={bid.confidence:.3f}'
        )
        print(
            f'     added distance={bid.detour_cost:.2f}, '
            f'time compliance={bid.time_compliance:.2f}'
        )

    selected_vehicle_id = request_agent.evaluate_bids(bids)
    negotiation_log.append(f'Vehicle selected: {selected_vehicle_id}')
    print(f'Selected vehicle: {selected_vehicle_id}')

    print('\nStep 3: The selected vehicle confirms the assignment.')
    assignment_success = request_agent.negotiate_assignment(
        selected_vehicle_id,
        system.vehicle_agents,
    )
    if assignment_success:
        print('Assignment confirmed.')
        negotiation_log.append('Assignment confirmed')

        assigned_vehicle = next(
            vehicle
            for vehicle in system.vehicle_agents
            if vehicle.id == selected_vehicle_id
        )
        if len(assigned_vehicle.assigned_requests) > 1:
            print(
                f'Load-balancing review suggested for {selected_vehicle_id}; '
                'no reassignment is performed in this simulation.'
            )
            negotiation_log.append('Load-balancing review noted')
        else:
            print('No load-balancing review is indicated.')
            negotiation_log.append('No load-balancing review indicated')
    else:
        print('Assignment was not confirmed.')
        negotiation_log.append('Assignment not confirmed')

    return {
        'urgent_request': urgent_request,
        'selected_vehicle': selected_vehicle_id,
        'assignment_success': assignment_success,
        'bids_received': len(bids),
        'negotiation_log': negotiation_log,
        'best_bids': sorted_bids,
    }


pharmacy_results = simulate_pharmaceutical_urgency_scenario(system)

## 9. Visualize the run

The plots summarize zone staffing, per-request local processing time, selected system metrics, and the urgent-request bid results. The success and failure categories are shown separately in the scatter plot so color has an explicit legend. Plot values are generated from the simulation objects rather than assumed in advance.

In [ ]:
def visualize_distributed_system_results(
    simulation_results: Dict,
    pharmacy_results: Dict,
    system: DistributedDVRPSystem,
) -> None:
    """Plot zone staffing, assignment runtime, metrics, and urgent-request bids."""
    figure, axes = plt.subplots(2, 2, figsize=(16, 12))
    figure.suptitle(
        'Distributed DVRP: Multi-Agent Coordination',
        fontsize=16,
        fontweight='bold',
    )

    ax1 = axes[0, 0]
    zone_names = [zone.id.replace('_', ' ').title() for zone in system.zone_agents]
    vehicle_counts = [len(zone.vehicles) for zone in system.zone_agents]
    bars = ax1.bar(
        zone_names,
        vehicle_counts,
        color='lightblue',
        alpha=0.8,
        edgecolor='black',
    )
    ax1.set_title('Vehicle Distribution by Zone')
    ax1.set_xlabel('Zone')
    ax1.set_ylabel('Number of vehicles')
    ax1.grid(True, alpha=0.3, axis='y')
    for bar, count in zip(bars, vehicle_counts):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(count),
            ha='center',
            va='bottom',
        )

    ax2 = axes[0, 1]
    processing_results = simulation_results['processing_results']
    processing_times_ms = [
        result['processing_time'] * 1000 for result in processing_results
    ]
    request_numbers = np.arange(1, len(processing_times_ms) + 1)
    successes = [result['assignment_success'] for result in processing_results]
    for assignment_success, color, label in [
        (True, 'green', 'Assigned'),
        (False, 'red', 'Not assigned'),
    ]:
        selected = [
            index
            for index, success in enumerate(successes)
            if success == assignment_success
        ]
        ax2.scatter(
            request_numbers[selected],
            np.asarray(processing_times_ms)[selected],
            color=color,
            alpha=0.75,
            label=label,
        )
    if processing_times_ms:
        average_time_ms = float(np.mean(processing_times_ms))
        ax2.axhline(
            y=average_time_ms,
            color='blue',
            linestyle='--',
            label=f'Mean: {average_time_ms:.3f} ms',
        )
    ax2.set_title('Local Request-Processing Time')
    ax2.set_xlabel('Request sequence number')
    ax2.set_ylabel('Processing time (ms)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    total_requests = simulation_results['requests_processed']
    success_rate = (
        simulation_results['successful_assignments'] / total_requests
        if total_requests
        else 0.0
    )
    metrics = {
        'Assignment rate': success_rate,
        'Assigned capacity share': system.system_metrics['vehicle_utilization'],
        'Mean processing time (s)': (
            system.system_metrics['average_response_time'] / 1000.0
        ),
    }
    metric_names = list(metrics.keys())
    metric_values = list(metrics.values())
    metric_bars = ax3.bar(
        metric_names,
        metric_values,
        color=['lightgreen', 'orange', 'lightcoral'],
        alpha=0.8,
        edgecolor='black',
    )
    ax3.set_title('System Metrics')
    ax3.set_ylabel('Value (units shown in labels)')
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.tick_params(axis='x', rotation=25)
    for bar, value in zip(metric_bars, metric_values):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f'{value:.4g}',
            ha='center',
            va='bottom',
        )

    ax4 = axes[1, 1]
    ax4.set_title('Urgent Pharmaceutical Request')
    ax4.set_xlabel('Bid rank')
    ax4.set_ylabel('Bid value (lower is preferred)')
    ax4.grid(True, alpha=0.3, axis='y')
    best_bids = pharmacy_results['best_bids']
    if best_bids:
        ax4.bar(
            [f'{rank}: {bid.vehicle_id}' for rank, bid in enumerate(best_bids, 1)],
            [bid.bid_value for bid in best_bids],
            color='mediumpurple',
            alpha=0.8,
            edgecolor='black',
        )
    else:
        ax4.text(
            0.5,
            0.5,
            'No feasible bids received',
            ha='center',
            va='center',
            transform=ax4.transAxes,
        )

    figure.tight_layout()
    plt.show()


visualize_distributed_system_results(
    simulation_results,
    pharmacy_results,
    system,
)

## 10. Analyze coordination patterns

The diagnostics describe workload spread, zone demand trends, assignment outcomes, and the share of assigned vehicle capacity. These are descriptive measures for the simulated batch, not guarantees of resilience or scalability. In particular, the workload balance coefficient divides the standard deviation of request counts by the mean; when the mean is zero, its denominator is set to one so the result remains defined.

In [ ]:
def analyze_emergent_optimization(
    system: DistributedDVRPSystem,
    simulation_results: Dict,
) -> Dict:
    """Summarize workload, demand, assignment, and capacity observations."""
    print('\nAnalysis of coordination patterns')
    print('=' * 40)

    analysis = {
        'load_balancing': {},
        'anticipatory_positioning': {},
        'adaptive_coordination': {},
        'system_resilience': {},
    }

    vehicle_workloads = [
        len(vehicle.assigned_requests) for vehicle in system.vehicle_agents
    ]
    workload_std = float(np.std(vehicle_workloads)) if vehicle_workloads else 0.0
    workload_mean = float(np.mean(vehicle_workloads)) if vehicle_workloads else 0.0
    analysis['load_balancing'] = {
        'mean_workload': workload_mean,
        'workload_std': workload_std,
        'balance_coefficient': workload_std / max(workload_mean, 1.0),
        'idle_vehicles': sum(workload == 0 for workload in vehicle_workloads),
        'overloaded_vehicles': sum(
            workload > workload_mean + workload_std
            for workload in vehicle_workloads
        ),
    }
    print('Load balancing:')
    print(f"  Mean requests per vehicle: {workload_mean:.3f}")
    print(f"  Workload standard deviation: {workload_std:.3f}")
    print(
        '  Balance coefficient: '
        f"{analysis['load_balancing']['balance_coefficient']:.3f}"
    )
    print(f"  Vehicles with no assignments: {analysis['load_balancing']['idle_vehicles']}")
    print(f"  Vehicles above the workload threshold: {analysis['load_balancing']['overloaded_vehicles']}")

    zone_forecasts = [zone.demand_forecast for zone in system.zone_agents]
    trends = [forecast.get('trend', 0.0) for forecast in zone_forecasts]
    current_demands = [forecast.get('current', 0) for forecast in zone_forecasts]
    analysis['anticipatory_positioning'] = {
        'zones_with_positive_trend': sum(trend > 0 for trend in trends),
        'average_demand_trend': float(np.mean(trends)) if trends else 0.0,
        'demand_variance': float(np.var(current_demands)) if current_demands else 0.0,
        'suggestions_recorded': sum(
            len(zone.repositioning_suggestions) for zone in system.zone_agents
        ),
    }
    print('\nAnticipatory positioning:')
    print(
        '  Zones with a positive demand trend: '
        f"{analysis['anticipatory_positioning']['zones_with_positive_trend']}"
    )
    print(
        '  Mean demand trend: '
        f"{analysis['anticipatory_positioning']['average_demand_trend']:.3f}"
    )
    print(
        '  Variance of current zone demand: '
        f"{analysis['anticipatory_positioning']['demand_variance']:.3f}"
    )
    print(
        '  Repositioning suggestions recorded: '
        f"{analysis['anticipatory_positioning']['suggestions_recorded']}"
    )

    total_requests = simulation_results['requests_processed']
    successful_assignments = simulation_results['successful_assignments']
    processing_results = simulation_results['processing_results']
    average_processing_time_ms = system.system_metrics['average_response_time']
    average_bids = (
        float(np.mean([result['bids_received'] for result in processing_results]))
        if processing_results
        else 0.0
    )
    success_rate = (
        successful_assignments / total_requests if total_requests else 0.0
    )
    analysis['adaptive_coordination'] = {
        'success_rate': success_rate,
        'avg_processing_time_ms': average_processing_time_ms,
        'bids_per_request': average_bids,
        'negotiation_efficiency': (
            success_rate / max(average_processing_time_ms / 1000.0, 1.0)
        ),
    }
    print('\nAssignment and local processing:')
    print(f"  Assignment rate: {success_rate:.1%}")
    print(f'  Mean local processing time: {average_processing_time_ms:.3f} ms')
    print(f'  Mean bids per request: {average_bids:.3f}')
    print(
        '  Descriptive rate per max(1 second, mean processing time): '
        f"{analysis['adaptive_coordination']['negotiation_efficiency']:.3f}"
    )

    active_vehicles = sum(workload > 0 for workload in vehicle_workloads)
    total_capacity = sum(vehicle.capacity for vehicle in system.vehicle_agents)
    used_capacity = sum(
        vehicle.capacity - vehicle.remaining_capacity
        for vehicle in system.vehicle_agents
    )
    analysis['system_resilience'] = {
        'active_vehicle_ratio': (
            active_vehicles / len(system.vehicle_agents)
            if system.vehicle_agents
            else 0.0
        ),
        'capacity_utilization': (
            used_capacity / total_capacity if total_capacity else 0.0
        ),
        'successful_assignments': successful_assignments,
        'vehicle_zone_product': len(system.vehicle_agents) * len(system.zone_agents),
    }
    print('\nSystem state:')
    print(
        '  Vehicles with at least one assignment: '
        f"{analysis['system_resilience']['active_vehicle_ratio']:.1%}"
    )
    print(
        '  Assigned capacity share: '
        f"{analysis['system_resilience']['capacity_utilization']:.1%}"
    )
    print(
        '  Successful assignments: '
        f"{analysis['system_resilience']['successful_assignments']}"
    )
    print(
        '  Vehicle-zone count product (descriptive only): '
        f"{analysis['system_resilience']['vehicle_zone_product']}"
    )
    return analysis


emergent_analysis = analyze_emergent_optimization(system, simulation_results)

## 11. Summarize the method and its limits

The final summary reports metrics calculated by the run and distinguishes demonstrated behavior from properties that this small simulation cannot establish. Local bidding can support geographically distributed decision-making, but the implementation does not prove linear scalability, fault tolerance, reduced delivery times, or global optimality. Those claims require appropriately designed experiments and, for operational use, a more complete routing and communication model.

In [ ]:
def distributed_intelligence_summary(
    system: DistributedDVRPSystem,
    simulation_results: Dict,
    pharmacy_results: Dict,
) -> None:
    """Report observed simulation results and the method's modeling boundaries."""
    requests_processed = simulation_results['requests_processed']
    successful_assignments = simulation_results['successful_assignments']
    success_rate = (
        successful_assignments / requests_processed
        if requests_processed
        else 0.0
    )

    print('\n' + '=' * 76)
    print('DYNAMIC VEHICLE ROUTING: DISTRIBUTED INTELLIGENCE SUMMARY')
    print('=' * 76)
    print('\nAgent roles:')
    print('  Vehicle agents calculate request bids and maintain local assignment state.')
    print('  Request agents solicit bids, select a vehicle, and confirm assignment.')
    print('  Zone agents forecast local request counts and record repositioning suggestions.')
    print('  Traffic conditions are supplied as input; no infrastructure agent is simulated.')

    print('\nAssignment protocol:')
    print('  Feasibility filtering is followed by weighted bidding and selection.')
    print('  Requests are processed sequentially, so earlier assignments affect later bids.')
    print('  The bid is a heuristic ranking score, not a globally optimized route cost.')

    print('\nObserved peak-hour run:')
    print(f'  Requests processed: {requests_processed}')
    print(f'  Successful assignments: {successful_assignments}')
    print(f'  Assignment rate: {success_rate:.1%}')
    print(
        '  Mean local processing time: '
        f"{system.system_metrics['average_response_time']:.3f} ms"
    )
    print(
        '  Assigned capacity share: '
        f"{system.system_metrics['vehicle_utilization']:.1%}"
    )

    print('\nUrgent-request run:')
    print(f"  Feasible bids received: {pharmacy_results['bids_received']}")
    print(f"  Selected vehicle: {pharmacy_results['selected_vehicle']}")
    print(f"  Assignment confirmed: {pharmacy_results['assignment_success']}")

    print('\nInterpretation and limitations:')
    print('  The run illustrates decentralized task assignment through local bids.')
    print('  Results depend on request order, feasibility, available capacity, and the heuristic weights.')
    print('  No centralized benchmark or global-optimality comparison is performed.')
    print('  Processing time is local code runtime, not end-to-end communication latency.')
    print('  Route insertion, load release after delivery, and vehicle repositioning are simplified.')
    print('  Real-world scalability and resilience require additional modeling and validation.')
    print('=' * 76)


distributed_intelligence_summary(system, simulation_results, pharmacy_results)